# CRTi: normalization, PCA and clustering

These excerpts retain the Scanpy normalization, variable-gene selection, PCA, UMAP and Leiden code used for the colorectal-cancer analysis. The working notebook contained additional exploratory sections; only the sections connected to this analysis are included here. Biological annotations, marker panels, sample comparisons and stored outputs are withheld.

Run `02-filtering-scrublet.ipynb` first. The public copy adds an explicit singlet selection before normalization. PCA and neighborhood choices remain visible as analysis settings; their suitability depends on the input. The notebook is a source-code portfolio and has not been rerun with the private dataset.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib import rcParams

# Run from the repository root. These local inputs and outputs are not distributed.
PRIVATE_DIR = Path("data/private/crti")
OUTPUT_DIR = Path("outputs/crti/python")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = OUTPUT_DIR
sc.settings.verbosity = 3
sc.settings.set_figure_params(scanpy=True, dpi=80, dpi_save=300,
                             frameon=True, vector_friendly=True, fontsize=14)
rcParams['figure.figsize'] = 8, 8

In [ ]:
adata = sc.read_h5ad(OUTPUT_DIR / 'qc-with-doublet-flags.h5ad')

In [ ]:
# Public adaptation: explicitly remove the doublets scored in notebook 02.
if adata.obs['Doublet'].isna().any():
    raise ValueError('Doublet predictions must be complete before singlet selection.')
adata = adata[~adata.obs['Doublet'].astype(bool), :].copy()
adata.layers['counts'] = adata.X.copy()

# Normalization

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
# Keep log-normalized all-gene expression before restricting to variable genes.
adata.raw = adata.copy()
adata = adata[:, adata.var['highly_variable']].copy()

## Scaling and PCA

The original workflow scaled variable genes before PCA. Regressing out count depth or mitochondrial fraction was considered separately; it is not applied in this excerpt.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack', n_comps=100)

In [ ]:
# Elbow plot
rcParams['figure.figsize'] = 15, 6
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=100, save="_pca-elbow.png")

## Neighborhood sensitivity

I compared neighbor graphs using different numbers of principal components before interpreting the clusters. These two retained examples overwrite the current graph and embedding in sequence; each figure has a distinct local filename. No cluster assignments or figures are embedded here.

In [ ]:
rcParams['figure.figsize'] = 8,8

sc.pp.neighbors(adata, knn=True, n_pcs=45)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1)
sc.pl.umap(adata, wspace = 0.2, size=30, color=["leiden", "Samples"], save="_leiden-pc45.png")

In [ ]:
rcParams['figure.figsize'] = 8,8

sc.pp.neighbors(adata, knn=True, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1)
sc.pl.umap(adata, wspace = 0.2, size=30, color=["leiden", "Samples"], save="_leiden-pc30.png")

## Marker-panel plotting interface

The original matrix-plot recipe is retained, with its study-specific gene list loaded from a private local text file (one gene per line). The marker list is not part of the repository.

In [ ]:
degs = (PRIVATE_DIR / 'marker-panel.txt').read_text().splitlines()
degs = [gene.strip() for gene in degs if gene.strip()]
if not degs:
    raise ValueError('Provide a nonempty private marker panel.')
sc.pl.matrixplot(adata, degs, 'leiden', dendrogram=False, cmap='viridis',
                 standard_scale='var', colorbar_title='Scaled\nExpression',
                 save='_marker-panel.png')

In [ ]:
adata.write(OUTPUT_DIR / 'clustered.h5ad')